# IAB Content Classifier — Colab end-to-end

Runs feature extraction and training on a Colab GPU runtime, persisting all outputs to Google Drive so a 12h session timeout doesn't lose work.

**Data layout:** videos are organized **by category folder** on Drive — `iab/<Tier1>/<Tier2>/.../<Leaf>/*.mp4` — where each folder is a taxonomy node name. There is **no hand-written `labels.csv`**: section 3 derives it from the folder tree.

**Before you start:**
1. Set runtime to GPU (Runtime → Change runtime type → T4 / L4 / A100).
2. Have your `iab/` folder tree on Drive at `/content/drive/MyDrive/iab/` (or edit `IAB_DRIVE`).
3. Set `REPO_URL` to your repo (must be public, or the clone will fail).

This notebook does a **test run on the beginning files** (`TEST_LIMIT`) so you can validate the pipeline end-to-end before committing to the full dataset.

## 1. Mount Drive, install deps, clone repo

In [2]:
import os
from google.colab import drive
drive.mount('/content/drive')

# Root of the tiered category tree on Drive (contains Attractions/ ... Food & Drink/).
IAB_DRIVE = '/content/drive/MyDrive/Colab Notebooks/iab'
REPO_URL  = 'https://github.com/matthewesp/iab-content-classifier.git'  # <-- set this
REPO_DIR  = '/content/iab-content-classifier'

# Persist HF model weights to Drive so DistilBERT / Whisper / DINO download
# only once across all sessions (~500MB total).
os.environ['HF_HOME'] = f'{IAB_DRIVE}/.hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
os.makedirs(f'{IAB_DRIVE}/cache', exist_ok=True)
os.makedirs(f'{IAB_DRIVE}/models', exist_ok=True)

!apt-get -qq install -y ffmpeg
# Note: errors are NOT silenced — if the clone fails (e.g. repo still private),
# you'll see it here instead of a confusing 'no pyproject.toml' later.
![ -d "$REPO_DIR/.git" ] && (cd "$REPO_DIR" && git pull -q) || git clone -q $REPO_URL "$REPO_DIR"
%cd $REPO_DIR
%pip install -q -e .

Mounted at /content/drive
/content/iab-content-classifier
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 80.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 37.7 MB/s eta 0:00:00
  Building editable for iab-content-classifier (pyproject.toml) ... done


## 2. Sanity check — device + ASR engine

On Colab GPU you should see `device: cuda` and `amp_enabled: True` (fp16 autocast on backbones). mlx-whisper is Apple-only and is skipped by the env marker in `pyproject.toml`.

In [3]:
import torch
from video_processor import get_device
print('device:', get_device())
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

# Lightweight backbone init to verify HF model downloads work + autocast is on
from multimodal_backbone import MultimodalBackbone
mb = MultimodalBackbone()
print('amp_enabled:', mb.amp_enabled)

device: cuda
cuda available: True
gpu: NVIDIA RTX PRO 6000 Blackwell Server Edition


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/100 [00:02<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: facebook/dino-vits16
Key                 | Status  | 
--------------------+---------+-
pooler.dense.bias   | MISSING | 
pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


amp_enabled: True


## 3. Generate `labels.csv` from the folder tree (sample across all data)

`scripts/build_labels_from_folders.py` walks `IAB_DRIVE`, maps each video's **deepest containing folder** back to its taxonomy Unique ID (matching the full root→leaf name path), and writes `video_path,leaf_id` rows.

For a test that **spans every category**, use `PER_LEAF` (first N videos per leaf) — this samples the whole taxonomy instead of just the alphabetically-first folders.

- `PER_LEAF = N` → N videos from **each** leaf (representative sample across all data).
- `TEST_LIMIT = N` → hard cap on total rows (applied after per-leaf).
- Set **both to None** for the full dataset.

`--abs` writes absolute Drive paths so `build_features.py` resolves them regardless of CWD.

In [4]:
# Sample across ALL categories: take the first N videos from every leaf folder.
PER_LEAF   = 25      # videos per leaf — bump up for a bigger test, None to disable
TEST_LIMIT = None   # optional hard cap on total rows; None = no cap

# Build the command in Python (robust to spaces in IAB_DRIVE and optional args)
# instead of relying on shell '$VAR' interpolation in the ! line.
cmd = (
    'python scripts/build_labels_from_folders.py '
    f'--iab "{IAB_DRIVE}" '
    '--taxonomy content_taxonomy_3.1.tsv '
    f'--out "{IAB_DRIVE}/labels.csv" --abs'
)
if PER_LEAF is not None:
    cmd += f' --per-leaf {PER_LEAF}'
if TEST_LIMIT is not None:
    cmd += f' --limit {TEST_LIMIT}'
print(cmd)
!{cmd}

import pandas as pd
_lbl = pd.read_csv(f'{IAB_DRIVE}/labels.csv')
print(f'{len(_lbl)} videos across {_lbl.leaf_id.nunique()} leaves')
_lbl.head(25)

python scripts/build_labels_from_folders.py --iab "/content/drive/MyDrive/Colab Notebooks/iab" --taxonomy content_taxonomy_3.1.tsv --out "/content/drive/MyDrive/Colab Notebooks/iab/labels.csv" --abs --per-leaf 25
Wrote 5928 labels across 238 leaves -> /content/drive/MyDrive/Colab Notebooks/iab/labels.csv
5928 videos across 238 leaves


,video_path,leaf_id
0,/content/drive/MyDrive/Colab Notebooks/iab/Att...,151
1,/content/drive/MyDrive/Colab Notebooks/iab/Att...,151
2,/content/drive/MyDrive/Colab Notebooks/iab/Att...,151
3,/content/drive/MyDrive/Colab Notebooks/iab/Att...,151
4,/content/drive/MyDrive/Colab Notebooks/iab/Att...,151
5,/content/drive/MyDrive/Colab Notebooks/iab/Att...,151
6,/content/drive/MyDrive/Colab Notebooks/iab/Att...,151
7,/content/drive/MyDrive/Colab Notebooks/iab/Att...,151
8,/content/drive/MyDrive/Colab Notebooks/iab/Att...,151
9,/content/drive/MyDrive/Colab Notebooks/iab/Att...,151


## 4. Feature extraction (the slow stage)

Extraction is **OCR + decode bound**, not GPU bound — the DINO/Whisper/DistilBERT backbones are ~10% of the time. So the throughput lever on the H100 is `--workers`: run many videos concurrently so OCR/decode overlap and the GPU stays fed. Each worker loads its own (small) backbone copy.

`WORKERS` ≈ start around the GPU/CPU sweet spot (8–12 on a Colab H100 box) and watch `videos/s`. The per-video cache writes to Drive, so an interrupted session resumes cheaply — rerun this cell and finished videos hit warm cache.

In [ ]:
WORKERS = 8   # concurrent video workers; tune for the H100/CPU box (try 8-12)

cmd = (
    'python scripts/build_features.py '
    f'--labels "{IAB_DRIVE}/labels.csv" '
    f'--out "{IAB_DRIVE}/features.pt" '
    '--taxonomy content_taxonomy_3.1.tsv '
    f'--cache-root "{IAB_DRIVE}/cache" '
    f'--workers {WORKERS}'
)
print(cmd)
!{cmd}

python scripts/build_features.py --labels "/content/drive/MyDrive/Colab Notebooks/iab/labels.csv" --out "/content/drive/MyDrive/Colab Notebooks/iab/features.pt" --taxonomy content_taxonomy_3.1.tsv --cache-root "/content/drive/MyDrive/Colab Notebooks/iab/cache" --workers 8
extracting with 8 workers (6 CPU threads each)…
Loading weights:   0% 0/100 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loading weights: 100% 100/100 [00:00<00:00, 11380.25it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different 

## 5. Train classifier

Reads `features.pt`, trains the root router + per-parent routers, writes weights to `$IAB_DRIVE/models/`. Sub-second per router on CUDA.

> On a tiny test slice many parents will have only 1-2 examples — training still runs, but accuracy numbers are meaningless until you scale up the labels. This step is mainly to confirm the train path works.

In [ ]:
!python scripts/train_classifier.py \
    --features "$IAB_DRIVE/features.pt" \
    --out-dir  "$IAB_DRIVE/models" \
    --taxonomy content_taxonomy_3.1.tsv \
    --epochs 50

## 6. (Optional) Smoke classify one video

Loads the trained routers from Drive and runs an end-to-end classification on the first labeled video.

In [ ]:
import json, pandas as pd
from pathlib import Path
from pipeline import VideoClassifier

clf = VideoClassifier(
    router_weights_dir=Path(f'{IAB_DRIVE}/models'),
    cache_root=Path(f'{IAB_DRIVE}/cache'),
)
video = Path(pd.read_csv(f'{IAB_DRIVE}/labels.csv').iloc[0]['video_path'])
print('video:', video.name)
print(json.dumps(clf.classify(video), indent=2))

## Notes

- **Folders are the labels.** Re-run section 3 whenever you add/move videos on Drive; it regenerates `labels.csv` from the current tree.
- **Scale up:** raise/remove `TEST_LIMIT` (or switch to `--per-leaf`) and rerun sections 3→5. Feature cache means already-processed videos are skipped.
- **Disconnects:** rerun cell 1 (re-mount + re-install — fast, wheels cached) then the section you were on. Cache hits skip processed videos.
- **HF downloads:** first feature-extraction run pulls DistilBERT + Whisper-tiny + DINO ViT-S/16 to `$HF_HOME` (~500MB, one-time per Drive).
- **Fast local training:** `features.pt` is small — download from Drive and run `train_classifier.py` on your M1; it's already sub-second.